# Milestone 1 🚩

In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [2]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [3]:
# CONFIGURATION
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems" # dataset path
GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]
# list of all genres available (alphabetical order)
STEMS = {'drums.wav', 'vocals.wav', 'bass.wav', 'other.wav'} # stems file name
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0

In [4]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    lessthan4kb = 0
    lessthan5mb = 0
    morethan5mb = 0

    rng = random.Random(seed)

    # ------------------- write your code here -------------------------------
    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        
        # Check if genre folder exists
        if not os.path.exists(genre_path):
            continue
            
        # List songs and sort them to ensure deterministic order before shuffling
        songs = sorted(os.listdir(genre_path))
        valid_songs_data = []

        for song_name in songs:
            song_dir = os.path.join(genre_path, song_name)
            if not os.path.isdir(song_dir):
                continue
            
            # Check Completeness and Corruption
            temp_paths = {}
            is_valid_song = True
            
            for stem in STEMS:
                stem_path = os.path.join(song_dir, stem)
                # Check 1: Completeness (file exists)
                # Check 2: Corruption (size >= 4KB / 4096 bytes)
                if not os.path.exists(stem_path):
                    is_valid_song = False
                    break
                if os.path.getsize(stem_path) < 4096:
                    lessthan4kb+=1
                    is_valid_song = False
                    break
                if os.path.getsize(stem_path) < 5.0491*1024*1024:
                    lessthan5mb+=1
                elif os.path.getsize(stem_path) > 5.0493*1024*1024:
                    morethan5mb+=1
                temp_paths[stem.replace('.wav', '')] = stem_path
            
            if is_valid_song:
                valid_songs_data.append(temp_paths)

        # Shuffle using the compulsory rng object
        rng.shuffle(valid_songs_data)

        # Stratified Split calculation
        split_idx = int(len(valid_songs_data) * (1 - val_split))
        train_list = valid_songs_data[:split_idx]
        val_list = valid_songs_data[split_idx:]

        # Helper function to populate dict
        def add_to_dict(target_dict, song_list):
            for song_map in song_list:
                for stem_key, path in song_map.items():
                    target_dict[genre][stem_key].append(path)
        
        add_to_dict(train_dataset, train_list)
        add_to_dict(val_dataset, val_list)
    #-------------------------------------------------------------------------
    print("less than 4 kb : ",lessthan4kb)
    print("less than 5 mb : ",lessthan5mb)
    print("more than 5mb : ",morethan5mb)
    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

less than 4 kb :  0
less than 5 mb :  1256
more than 5mb :  184


Complete the function `build_dataset` in question notebook and answer following questions (Q1 to Q3).  
Hint: 1kb = 1024 bytes  

**Q1)** What is the value of total number of corrupted sounds ( less than 4kb) + (total number of sounds < 5.0491MB)

**Ans** : 1256

**Q2)** What is the absolute difference between  total number of sounds > 5.0493MB and total number of sounds < 5.0491MB ? 

In [5]:
1256-184

1072

**Q3)** What is the absolute difference between the number of training reggae drum samples and the number of validation country vocal samples? 

In [6]:
len(tr["reggae"]["drums"])-len(val["country"]["vocals"])

66

Complete the function `find_long_silences` in question notebook and answer following questions (Q4 to Q9).

In [8]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
    Output:
        df: Pandas DataFrame containing details of all files with silence >= 5s
    """
    records = []
    # ------------------- write your code here -------------------------------

    total_files = sum(len(paths) for genre in dataset_dict.values() for paths in genre.values())
    
    # Iterate through the nested dictionary
    for genre, stems in dataset_dict.items():
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                
                # Load Audio
                try:
                    y, _ = librosa.load(file_path, sr=sr)
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
                    continue
                    
                total_duration = len(y) / sr
                
                # Find Non-Silent Intervals
                # intervals is a list of [start_sample, end_sample]
                intervals = librosa.effects.split(y, top_db=top_db)
                
                silence_locations = []
                max_silence = 0.0

                # CASE A: Fully silent
                if len(intervals) == 0:
                    max_silence = total_duration
                    if max_silence >= threshold_sec:
                        silence_locations.append("Full")

                else:
                    # CASE B: START silence
                    # Gap from 0 to the start of the first interval
                    start_silence_samples = intervals[0][0]
                    start_silence_sec = start_silence_samples / sr
                    
                    if start_silence_sec > max_silence:
                        max_silence = start_silence_sec
                    
                    if start_silence_sec >= threshold_sec:
                        silence_locations.append("Start")

                    # CASE C: END silence
                    # Gap from the end of the last interval to total duration
                    end_silence_samples = len(y) - intervals[-1][1]
                    end_silence_sec = end_silence_samples / sr
                    
                    if end_silence_sec > max_silence:
                        max_silence = end_silence_sec
                        
                    if end_silence_sec >= threshold_sec:
                        silence_locations.append("End")

                    # CASE D: MIDDLE silence
                    # Gaps between consecutive intervals
                    found_middle = False
                    for i in range(len(intervals) - 1):
                        current_end = intervals[i][1]
                        next_start = intervals[i+1][0]
                        gap_sec = (next_start - current_end) / sr
                        
                        if gap_sec > max_silence:
                            max_silence = gap_sec
                        
                        if gap_sec >= threshold_sec:
                            found_middle = True
                    
                    if found_middle:
                        silence_locations.append("Middle")

                # Store result if threshold met
                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre,
                        "Stem": stem_name,
                        "Duration": round(total_duration, 2),
                        "Max_Silence_Sec": round(max_silence, 2),
                        "Silence_Location": ", ".join(silence_locations),
                        "File_Path": file_path
                    })
    #-------------------------------------------------------------------------
    df = pd.DataFrame(records)
    return df


# --- EXECUTION ---
# Pass your 'tr' (training) dictionary here.
# Ensure 'tr' is defined from your previous build_dataset code.
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

**Q4)** Total number of sound files having silence greater than equal to 5 secs

In [9]:
len(df_silence)

680

**Q5)** Total number of sound tracks in Vocals where silence >= 5 secs

In [10]:
len(df_silence[df_silence['Stem'] == 'vocals'])

304

**Q6)** What's the average Silence Length in Vocals (in secs).

In [11]:
df_silence[df_silence['Stem'] == 'vocals']['Max_Silence_Sec'].mean()

np.float64(12.590789473684211)

**Q7)** Total number of drums sound tracks in jazz where silence >= 5 secs

In [12]:
len(df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums')])

24

**Q8)** Total number of drums sound tracks in jazz where silence >= 5 secs and Silence_Location is only middle.

In [13]:
len(df_silence[
    (df_silence['Genre'] == 'jazz') & 
    (df_silence['Stem'] == 'drums') & 
    (df_silence['Silence_Location'] == 'Middle')
])

13

**Q9)** Total number of drums sound tracks in jazz where silence >= 5 secs and Max_Silence_Sec >= 10.

In [14]:
len(df_silence[
    (df_silence['Genre'] == 'jazz') & 
    (df_silence['Stem'] == 'drums') & 
    (df_silence['Max_Silence_Sec'] >= 10)
])

7

Select the first song from the ‘rock’ genre, combine all stems to prepare a sample, perform the tasks outlined in the notebook, and answer questions 10–12 based on the results.

In [15]:
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0

In [16]:
stems_audio = []
try:
    for key in STEM_KEYS:
        path = tr[GENRE_TO_TEST][key][SONG_INDEX]
        print(path)
        y, _ = librosa.load(path, sr=SR, duration=5.0)
        stems_audio.append(y)
    # ------------------- write your code here -------------------------------
    # Load audio (Duration 5.0s for speed/consistency)
    #-------------------------------------------------------------------------

    print("Audio loaded successfully.")
except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")


/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00092/drums.wav
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00092/vocals.wav
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00092/bass.wav
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00092/other.wav
Audio loaded successfully.


In [17]:
# ------------------- write your code here -------------------------------
# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.array(stems_audio)

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)

# Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(np.square(mix_raw)))

#Peak Normalization
max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
#------------------------------------------------------------------------

**Q10)** What is the length of the mix sample?

In [18]:
len(mix_raw)

110250

**Q11)** What is the value of RMS Amplitude of mix sample?

In [19]:
rms_val

np.float32(0.102124155)

**Q12)** What is the value of max value of peak  normalized sample ?

In [20]:
max_val

np.float32(0.58938766)